# Contextual GP: SACGP / LCEAGP / LCEMGP

## 1. Contextual GPとは

Contextual GPは、複数のコンテキストに分かれた構造を利用するGPです。

このNotebookでは2つの工程コンテキストを考えます。

- `context_A`: 条件 `[a0, a1]`
- `context_B`: 条件 `[b0, b1]`

`SACGP` と `LCEAGP` は、コンテキスト別の報酬を直接観測せず、**合計された報酬**だけを観測するケースを扱います。

`LCEMGP` は逆に、各コンテキストの報酬を個別に観測できるmulti-output / multi-task型です。

### このNotebookで学ぶこと

- **集約報酬しか観測できない場合**と**context別の報酬が観測できる場合**の違い
- SACGP・LCEAGP・LCEMGPのモデル化の前提
- `decomposition`、`task_feature`、`context_cat_feature` の役割
- posteriorの予測平均と不確実性の可視化
- contextを考慮した候補提案と観測データ更新

contextは工程、装置、部位など、応答の構造を分ける意味のある単位を想定します。単なるカテゴリ変数やMulti-taskのtask IDとは役割を区別してください。

In [ ]:
import matplotlib.pyplot as plt
import torch

from robotorchan.models import LCEAGP, LCEMGP, SACGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. SACGP / LCEAGP用の集約報酬データ

### 観測できるデータによるモデル選択

例えば製品の総合品質スコアが、工程Aと工程Bの寄与の**和**として決まるとします。工程別の品質を測定できず、最終スコアだけ得られる場合は、`SACGP` または `LCEAGP` の集約報酬モデルが候補になります。

一方、工程Aと工程Bのスコアを**別々に観測**できる場合は、context別の出力を学習する `LCEMGP` が候補になります。後者は単に集約報酬のモデルを入れ替えたものではなく、学習時の観測形式も異なります。

In [ ]:
def reward_a(X):
    return torch.sin(2 * torch.pi * X[:, 0]) + 0.4 * X[:, 1]

def reward_b(X):
    return 0.7 * torch.cos(2 * torch.pi * X[:, 0]) - 0.3 * X[:, 1]

n = 24
train_X = torch.rand(n, 4)

y_a = reward_a(train_X[:, [0, 1]])
y_b = reward_b(train_X[:, [2, 3]])
train_Y = (y_a + y_b + 0.03 * torch.randn(n)).unsqueeze(-1)

decomposition = {
    "context_A": [0, 1],
    "context_B": [2, 3],
}

print(train_X.shape, train_Y.shape)

### `decomposition` とデータshape

入力は `[a0, a1, b0, b1]` の4列です。`decomposition={"context_A": [0, 1], "context_B": [2, 3]}` は、どの列がどのcontextに属するかを示します。

`train_X.shape == (24, 4)`、`train_Y.shape == (24, 1)` であり、**観測したのは各contextの個別報酬ではなく、2つを足したスカラー報酬**です。合成関数 `reward_a` と `reward_b` は説明用の真値で、実務では内訳を観測できない状況を想定します。

## 3. SACGP

SACGPは各コンテキストを独立な加法成分として扱います。
コンテキスト間に明示的な相関を学習させない、比較的単純な構造です。

In [ ]:
sac = SACGP(
    train_X=train_X,
    train_Y=train_Y,
    train_Yvar=None,
    decomposition=decomposition,
)

print("supports_mll:", sac.supports_mll)
print("supports_fit:", sac.supports_fit)
print("raw_train_X:", sac.raw_train_X.shape)
print("raw_train_Y:", sac.raw_train_Y.shape)

sac.fit()
sac.eval()

### SACGPの加法性仮定

SACGPは概念的に $f(x)=f_A(x_A)+f_B(x_B)$ の形を仮定します。各contextの寄与を加法的に表現するため、context間に強い相互作用がある場合には適合しない可能性があります。

ここでは個別の $f_A$ と $f_B$ を直接観測せず、合計値で学習します。したがって、合計の予測精度と個別成分の識別可能性は別の問題です。

## 4. LCEAGP

LCEAGPは潜在context embeddingを学習し、コンテキスト間の類似性・相関を表現します。
`cat_feature_dict=None` の場合はcontext名に対応する内部カテゴリ表現が使われます。

In [ ]:
lcea = LCEAGP(
    train_X=train_X,
    train_Y=train_Y,
    train_Yvar=None,
    decomposition=decomposition,
    train_embedding=True,
)

print("supports_mll:", lcea.supports_mll)
print("supports_fit:", lcea.supports_fit)
print("raw_train_X:", lcea.raw_train_X.shape)
print("raw_train_Y:", lcea.raw_train_Y.shape)

lcea.fit()
lcea.eval()

### LCEAGPのembedding

LCEAGPはcontext間の類似性を潜在embeddingで表します。`train_embedding=True` はembeddingを学習する設定です。

SACGPより表現力が高い可能性がある一方、少量データではembeddingの推定が不安定になることもあります。**相関を学習することが必ず精度向上につながるわけではありません**。

## 5. SACGPとLCEAGPのposterior比較

In [ ]:
grid = torch.linspace(0.0, 1.0, 120)
test_X = torch.stack(
    [
        grid,
        torch.full_like(grid, 0.5),
        grid,
        torch.full_like(grid, 0.5),
    ],
    dim=-1,
)

with torch.no_grad():
    sac_post = sac.posterior(test_X)
    lcea_post = lcea.posterior(test_X)

sac_mean = sac_post.mean.squeeze(-1)
lcea_mean = lcea_post.mean.squeeze(-1)

truth = (
    reward_a(test_X[:, [0, 1]])
    + reward_b(test_X[:, [2, 3]])
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(grid, truth, label="真値")
plt.plot(grid, sac_mean, label="SACGP")
plt.plot(grid, lcea_mean, label="LCEAGP")
plt.xlabel("共通走査値")
plt.ylabel("集約報酬")
plt.legend()
plt.show()

### 追加実装例：集約報酬の不確実性を比較

既存のグラフは予測平均だけを表示しています。次はSACGPとLCEAGPそれぞれの事後分散から、**潜在集約報酬の近似95%区間**を表示します。

これは `a0=b0` を同時に走査し、他の入力を0.5に固定した断面です。4次元空間全体の予測性能を表すものではなく、観測ノイズを含む将来測定値の予測区間でもありません。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, name, post in [
    (axes[0], "SACGP", sac_post),
    (axes[1], "LCEAGP", lcea_post),
]:
    prediction = post.mean.squeeze(-1).detach()
    uncertainty = post.variance.clamp_min(0).sqrt().squeeze(-1).detach()
    ax.plot(grid, truth, "--", label="true aggregate reward")
    ax.plot(grid, prediction, label="posterior mean")
    ax.fill_between(grid, prediction - 1.96 * uncertainty,
                    prediction + 1.96 * uncertainty,
                    alpha=0.2, label="approx. 95% latent interval")
    ax.set_title(name)
    ax.set_xlabel("shared sweep value")
    ax.legend(fontsize=8)
axes[0].set_ylabel("aggregate reward")
plt.tight_layout()
plt.show()

## 6. LCEMGP: コンテキスト別報酬が観測できる場合

LCEMGPでは、各contextをtaskとしてlong-formatで表現します。

ここでは共通の決定変数 `[x0, x1]` に対して、

- task 0 = context_A
- task 1 = context_B

とします。

### LCEMGPのlong-formatデータ

LCEMGPでは同じ決定変数 `[x0, x1]` にtask IDを追加し、context AとBの観測を**行方向に積み重ねる**long-formatで学習します。

`mt_X.shape == (40, 3)`、`mt_Y.shape == (40, 1)` です。`task_feature=2` は3列目をcontext識別に使う指定で、`context_cat_feature` はcontextの属性表現です。集約報酬用の4列入力とはデータ構造が異なります。

In [ ]:
base_X = torch.rand(20, 2)

X_a = torch.cat([base_X, torch.zeros(20, 1)], dim=-1)
X_b = torch.cat([base_X, torch.ones(20, 1)], dim=-1)

Y_a = (reward_a(base_X) + 0.03 * torch.randn(20)).unsqueeze(-1)
Y_b = (reward_b(base_X) + 0.03 * torch.randn(20)).unsqueeze(-1)

mt_X = torch.cat([X_a, X_b], dim=0)
mt_Y = torch.cat([Y_a, Y_b], dim=0)

context_cat_feature = torch.tensor([
    [0.0],
    [1.0],
])

lcem = LCEMGP(
    train_X=mt_X,
    train_Y=mt_Y,
    task_feature=2,
    context_cat_feature=context_cat_feature,
)

print("supports_mll:", lcem.supports_mll)
print("supports_fit:", lcem.supports_fit)
print("raw_train_X:", lcem.raw_train_X.shape)
print("raw_train_Y:", lcem.raw_train_Y.shape)

lcem.fit()
lcem.eval()

## 7. LCEMGPのcontext別posterior

### LCEMGPのposterior

`posterior(test_base, output_indices=[0, 1])` はcontext AとBを**別々の出力**として予測します。`mean[..., 0]` がA、`mean[..., 1]` がBです。

両出力は同じ入力条件で比較できますが、SACGP・LCEAGPの集約報酬とそのまま同一視しないでください。

In [ ]:
test_base = torch.stack(
    [grid, torch.full_like(grid, 0.5)],
    dim=-1,
)

with torch.no_grad():
    post = lcem.posterior(test_base, output_indices=[0, 1])
    mean = post.mean

mean_a = mean[..., 0]
mean_b = mean[..., 1]
truth_a = reward_a(test_base)
truth_b = reward_b(test_base)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(grid, truth_a, label="context_A 真値")
plt.plot(grid, mean_a, "--", label="context_A LCEMGP")
plt.plot(grid, truth_b, label="context_B 真値")
plt.plot(grid, mean_b, "--", label="context_B LCEMGP")
plt.xlabel("x0（x1=0.5で固定）")
plt.ylabel("context別報酬")
plt.legend()
plt.show()

### 追加実装例：context別の不確実性

LCEMGPについても、contextごとの予測平均と約95%の潜在関数区間を描画します。各contextのデータが十分でない領域では、区間が広くなる可能性があります。

In [ ]:
with torch.no_grad():
    context_std = post.variance.clamp_min(0).sqrt()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for index, (ax, target, label) in enumerate([
    (axes[0], truth_a, "context_A"),
    (axes[1], truth_b, "context_B"),
]):
    prediction = mean[..., index].detach()
    uncertainty = context_std[..., index].detach()
    ax.plot(grid, target, "--", label="true reward")
    ax.plot(grid, prediction, label="posterior mean")
    ax.fill_between(grid, prediction - 1.96 * uncertainty,
                    prediction + 1.96 * uncertainty,
                    alpha=0.2, label="approx. 95% latent interval")
    ax.set_title(label)
    ax.set_xlabel("x0")
    ax.legend(fontsize=8)
axes[0].set_ylabel("context reward")
plt.tight_layout()
plt.show()

## 8. 3モデルの使い分け

| モデル | 観測する出力 | context間の関係 |
|---|---|---|
| SACGP | 集約された1つの報酬 | 独立な加法構造 |
| LCEAGP | 集約された1つの報酬 | 潜在embeddingで相関を学習 |
| LCEMGP | context別の報酬 | multi-taskとしてcontext相関を学習 |

### 選択の目安

- context別の内訳が観測できない → `SACGP` または `LCEAGP`
- context同士の類似性を学習したい → `LCEAGP`
- context別の出力を直接観測できる → `LCEMGP`

Contextual GPは、単にカテゴリ変数があるだけの問題とは異なります。
「複数contextの構造を明示的に利用できるか」が採用判断のポイントです。

## Context-aware candidate proposals

SACGPとLCEAGPは集約報酬、LCEMGPはcontext別報酬を用いて候補を選択します。ここでは既存の候補グリッド上で事後平均を最大化します。

### モデル選択で確認する点

最初に確認すべきことは、**context別の内訳を観測できるか**です。集約報酬だけならSACGP・LCEAGP、内訳を直接観測できるならLCEMGPが候補になります。

次に、context間の相関をモデル化する必要性、データ量、計算コストを比較します。モデルの表現力が高いほど必ず良いわけではありません。

In [ ]:
def contextual_objective(X):
    return (reward_a(X[:, [0, 1]]) + reward_b(X[:, [2, 3]])).unsqueeze(-1)

for name, surrogate in [("SACGP", sac), ("LCEAGP", lcea)]:
    with torch.no_grad():
        scores = surrogate.posterior(test_X).mean.squeeze(-1)
    index = scores.argmax()
    candidate = test_X[index:index + 1]
    assert candidate.shape == (1, 4)
    print(name, "candidate:", candidate, "objective:", contextual_objective(candidate))

with torch.no_grad():
    context_scores = lcem.posterior(test_base, output_indices=[0, 1]).mean
for context_index, reward in enumerate((reward_a, reward_b)):
    index = context_scores[:, context_index].argmax()
    candidate = test_base[index:index + 1]
    assert candidate.shape == (1, 2)
    print("context", context_index, "candidate:", candidate, "objective:", reward(candidate))


## Mixed LCEMGP: カテゴリを含むcontext別の候補提案

カテゴリ列とtask列を区別します。観測済みカテゴリ値を保持した候補プールを評価し、各contextの事後平均が最大の候補を選択します。


### 候補提案の位置付け

既存の例は有限の候補グリッドで**事後平均を最大化するgreedy探索**です。EI/UCBのような予測不確実性を使う獲得関数最適化とは異なります。

SACGP・LCEAGPは**集約報酬**を最大化し、LCEMGPは**各contextの報酬**を別々に最大化します。目的関数が違うので、提案点や評価値を直接比較するときには注意が必要です。

In [ ]:
from robotorchan.models import MixedLCEMGP

mixed_base_X = torch.cat(
    [base_X, torch.arange(base_X.shape[0]).remainder(2)
     .to(base_X).unsqueeze(-1)],
    dim=-1,
)
mixed_a_X = torch.cat(
    [mixed_base_X, torch.zeros(base_X.shape[0], 1)], dim=-1
)
mixed_b_X = torch.cat(
    [mixed_base_X, torch.ones(base_X.shape[0], 1)], dim=-1
)
mixed_context_X = torch.cat([mixed_a_X, mixed_b_X], dim=0)
mixed_a_Y = (
    reward_a(base_X) + 0.1 * mixed_base_X[:, 2]
).unsqueeze(-1)
mixed_b_Y = (
    reward_b(base_X) + 0.1 * mixed_base_X[:, 2]
).unsqueeze(-1)
mixed_context_Y = torch.cat([mixed_a_Y, mixed_b_Y], dim=0)
mixed_lcem = MixedLCEMGP(
    mixed_context_X, mixed_context_Y, task_feature=3,
    cat_dims=[2], context_cat_feature=context_cat_feature,
)
mixed_lcem.fit()
mixed_lcem.eval()
mixed_candidates = torch.cat(
    [test_base.repeat(2, 1),
     torch.arange(2).repeat_interleave(test_base.shape[0])
     .to(test_base).unsqueeze(-1)],
    dim=-1,
)
with torch.no_grad():
    mixed_scores = mixed_lcem.posterior(
        mixed_candidates, output_indices=[0, 1]
    ).mean
for context_index, reward in enumerate((reward_a, reward_b)):
    selected = mixed_scores[:, context_index].argmax()
    candidate = mixed_candidates[selected:selected + 1]
    observed = reward(candidate[:, :2]) + 0.1 * candidate[:, 2]
    assert candidate[0, 2].item() in (0.0, 1.0)
    print("Mixed LCEMGP", context_index, candidate, observed)


## Hierarchical conditional GP: 有効な枝の候補だけを選択

親変数 `process_type` によって有効なパラメータが変わります。候補プールは枝0で温度、枝1で圧力だけを走査し、無効な変数は0.5に固定します。Mixed版では追加のカテゴリ列を持ちます。


### Mixed LCEMGPのカテゴリとcontext

Mixed版では、連続設計変数2列にカテゴリ設計変数を追加し、さらにcontextをtaskとして扱います。`cat_dims=[2]` はカテゴリ列、`task_feature=3` はcontext識別列です。

**カテゴリ設計変数は探索対象、context/taskは予測する出力の指定**です。両者を混同しないようにしてください。

In [ ]:
from robotorchan.models import (
    HierarchicalConditionalKernelGP,
    MixedHierarchicalConditionalKernelGP,
)

hierarchical_dependencies = {1: {0: [2], 1: [3]}}
hierarchical_X = torch.cat(
    [
        torch.cat(
            [base_X, torch.zeros(base_X.shape[0], 1), base_X[:, :1]],
            dim=-1,
        ),
        torch.cat(
            [base_X, torch.ones(base_X.shape[0], 1), base_X[:, :1]],
            dim=-1,
        ),
    ],
    dim=0,
)
hierarchical_X[:, 2] = hierarchical_X[:, 1]
hierarchical_Y = (
    torch.sin(2 * torch.pi * hierarchical_X[:, :1])
    + torch.where(
        hierarchical_X[:, 1:2] == 0,
        0.8 * (hierarchical_X[:, 2:3] - 0.5),
        -0.7 * (hierarchical_X[:, 3:4] - 0.5),
    )
)
hierarchical_mixed_X = torch.cat(
    [
        hierarchical_X,
        torch.arange(hierarchical_X.shape[0]).remainder(2)
        .to(hierarchical_X).unsqueeze(-1),
    ],
    dim=-1,
)
hierarchical_mixed_Y = hierarchical_Y + 0.1 * hierarchical_mixed_X[:, 4:5]
hierarchical_models = [
    (
        "Hierarchical",
        HierarchicalConditionalKernelGP(
            hierarchical_X, hierarchical_Y,
            hierarchical_dependencies=hierarchical_dependencies,
            use_saas_prior=False,
        ),
        False,
    ),
    (
        "Mixed hierarchical",
        MixedHierarchicalConditionalKernelGP(
            hierarchical_mixed_X, hierarchical_mixed_Y,
            hierarchical_dependencies=hierarchical_dependencies,
            cat_dims=[4], use_saas_prior=False,
        ),
        True,
    ),
]
branch_grid = torch.linspace(0.0, 1.0, 40)
branch0 = torch.stack(
    [branch_grid, torch.zeros_like(branch_grid), branch_grid,
     torch.full_like(branch_grid, 0.5)],
    dim=-1,
)
branch1 = torch.stack(
    [branch_grid, torch.ones_like(branch_grid),
     torch.full_like(branch_grid, 0.5), branch_grid],
    dim=-1,
)
branch_pool = torch.cat([branch0, branch1], dim=0)
for name, surrogate, is_mixed in hierarchical_models:
    surrogate.fit()
    surrogate.eval()
    if is_mixed:
        pool = torch.cat(
            [
                torch.cat([branch_pool, torch.zeros(80, 1)], dim=-1),
                torch.cat([branch_pool, torch.ones(80, 1)], dim=-1),
            ],
            dim=0,
        )
    else:
        pool = branch_pool
    with torch.no_grad():
        scores = surrogate.posterior(pool).mean.squeeze(-1)
    candidate = pool[scores.argmax():scores.argmax() + 1]
    observed = (
        torch.sin(2 * torch.pi * candidate[:, :1])
        + torch.where(
            candidate[:, 1:2] == 0,
            0.8 * (candidate[:, 2:3] - 0.5),
            -0.7 * (candidate[:, 3:4] - 0.5),
        )
    )
    if is_mixed:
        observed = observed + 0.1 * candidate[:, 4:5]
    assert candidate[0, 1].item() in (0.0, 1.0)
    assert torch.isfinite(observed).all()
    print(name, "candidate:", candidate, "objective:", observed)


### 階層型GPとの違い

Contextual GPは、複数contextの報酬の集約や相関をモデル化します。階層型GPは、親条件に応じて**有効な子変数が変わる探索空間**をモデル化します。

このNotebook末尾の階層型GPは関連モデルの参考例です。contextの分解構造と条件付き有効性は、同じ概念ではありません。

## 9. 追加実装例：集約報酬の新しい観測を追加

SACGPが提案する候補を1点評価した場合、観測できるのはcontext A/Bの内訳ではなく**合計報酬1点**です。次のコードではその観測を追加します。

計算時間を抑えるため再学習は実行しません。実際のBOループでは、更新したデータを使ってSACGPなどを再構築・学習し、次の候補を選びます。

In [ ]:
with torch.no_grad():
    sac_scores = sac.posterior(test_X).mean.squeeze(-1)
selected_index = int(sac_scores.argmax().item())
new_X = test_X[selected_index:selected_index + 1]
new_Y = contextual_objective(new_X)
updated_X = torch.cat([train_X, new_X], dim=0)
updated_Y = torch.cat([train_Y, new_Y], dim=0)
assert new_X.shape == (1, 4)
assert new_Y.shape == (1, 1)
assert updated_X.shape == (n + 1, 4)
assert updated_Y.shape == (n + 1, 1)
print("selected design:", new_X)
print("observed aggregate reward:", new_Y)
print("updated data shapes:", updated_X.shape, updated_Y.shape)